In [ ]:
import numpy as np

data = np.genfromtxt("timing_summary.csv", delimiter=",", names=True)
Larr = np.unique(data["L"]).astype(int)
totalruns = int(data["runs"].sum())
x = Larr**2
# Preserve the original ten-slot normalization.
y = np.array([data["mean_seconds"][data["L"] == size].sum() / 10 for size in Larr])
yerror = np.array([
    np.sqrt(np.square(data["historical_batch_uncertainty"][data["L"] == size]).sum() / 10)
    for size in Larr
])


In [ ]:
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

order = np.argsort(x)
x = x[order]
y = y[order]
yerror = yerror[order]

fit_start = 1
xfit = x[fit_start:]
yfit = y[fit_start:]
yerrfit = yerror[fit_start:]

def frac_residuals(xdata, A, alpha, C):
    ypred = A * xdata**alpha + C
    return yfit / ypred - 1

target = np.zeros(len(yfit))
sigma_frac = yerrfit / yfit

p0 = [yfit[-1] / xfit[-1], 1.0, 0.0]

bounds = (
    [0.0, 0.0, 0.0],
    [np.inf, 3.0, np.inf]
)

popt, pcov = curve_fit(
    frac_residuals,
    xfit,
    target,
    p0=p0,
    bounds=bounds,
    # sigma=sigma_frac,
    # absolute_sigma=True,
    maxfev=10000
)

A, power, C = popt
uncertainties = np.sqrt(np.diag(pcov))

def model(xdata):
    return A * xdata**power + C

print(f"Fit Power N^{power:.4f} +- {uncertainties[1]:.3e}")

xline = np.logspace(np.log10(x.min()), np.log10(x.max()), 300)
yline = model(xline)
ybest = model(x)

plt.figure()
plt.plot(xline, yline, label=fr"fit: $t \propto N^{{{power:.3f} \pm {uncertainties[1]:.3f}}}$")
plt.errorbar(x, y, yerr=1e3*yerror, fmt='o', label=r"data, uncertainties $\times 1000$")
plt.yscale("log")
plt.xscale("log")
plt.xlabel("Total Number of Lattice Points N")
plt.ylabel("Time (s)")
plt.title(f"Efficiency of Newman-Ziff Algorithm, total runs:{totalruns}")
plt.legend()
plt.show()

frac_resid = y / ybest - 1

plt.figure()
plt.errorbar(
    x[fit_start:],
    frac_resid[fit_start:],
    yerr=10 * np.abs(yerror[fit_start:] / y[fit_start:]),
    fmt='o',
    label=r"fit residuals, fractional uncertainties $\times 10$"
)
plt.axhline(0, linestyle="--")
plt.xscale("log")
plt.xlabel("N")
plt.ylabel("Fractional residual: data / fit - 1")
plt.title(f"Fractional Residuals of Fit, total runs:{totalruns}")
plt.legend()
plt.show()

plt.figure()
plt.scatter(x, yerror / y)
plt.yscale("log")
plt.xscale("log")
plt.xlabel("N")
plt.ylabel("Fractional Uncertainties")
plt.title("Fractional Uncertainties")
plt.show()